In [2]:
# Cell 1: Install dependencies from bundled wheels (no internet needed)
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
    "--find-links", "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    "arc_agi", "arcengine", "pydantic", "numpy", "pillow", "python_dotenv", "requests"
])

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


0

In [11]:

# Cell 2: Copy code to writable directory and set up paths
import os, shutil, sys

MODEL = "/kaggle/input/models/jazmiahenry/cascading-cognition-engine/other/default/1" 
COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
WORK = "/kaggle/working"

# Copy agents from competition data
src = os.path.join(COMP, "ARC-AGI-3-Agents/agents")
dst = os.path.join(WORK, "agents")
if os.path.exists(dst):
    shutil.rmtree(dst)                                                        
shutil.copytree(src, dst)
                                                                                
# Copy cognitive module from your model   
src = os.path.join(MODEL, "cognitive")
dst = os.path.join(WORK, "cognitive")                                         
if os.path.exists(dst):
    shutil.rmtree(dst)                                                        
shutil.copytree(src, dst)                 

# Copy your cognitive_agent.py into agents/templates/
shutil.copy(os.path.join(MODEL, "cognitive_agent.py"), os.path.join(WORK,"agents/templates/cognitive_agent.py"))                                       
   
# Overwrite agents/__init__.py to only load our agent                         
init_code = (                             
    "from typing import Type, cast\n"
    "from .agent import Agent, Playback\n"                                    
    "from .recorder import Recorder\n"
    "from .swarm import Swarm\n"                                              
    "from .templates.cognitive_agent import CognitiveAgent\n"
    "AVAILABLE_AGENTS = {'cognitiveagent': CognitiveAgent}\n"
)                                                                             
with open(os.path.join(WORK, "agents", "__init__.py"), "w") as f:
    f.write(init_code)                                                        
                                            
os.makedirs(os.path.join(WORK, "recordings"), exist_ok=True)                  
os.environ["OPERATION_MODE"] = "offline"
os.environ["ENVIRONMENTS_DIR"] = os.path.join(COMP, "environment_files")      
os.environ["RECORDINGS_DIR"] = os.path.join(WORK, "recordings")               
os.environ["ARC_BASE_URL"] = "https://three.arcprize.org/"
os.environ["DEBUG"] = "False"                                                 
os.environ["TESTING"] = "False"           
                                                                                
if WORK not in sys.path:
    sys.path.insert(0, WORK)                                                  
                                            
print("Setup complete.")

Setup complete.


In [ ]:
import json, logging, sys, os
from agents.swarm import Swarm
                                                                                
logger = logging.getLogger()
logger.setLevel(logging.INFO)                                                 
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s |%(message)s"))
if not logger.handlers:
    logger.addHandler(handler)                                                
   
env_dir = os.environ["ENVIRONMENTS_DIR"]                                      
games = []                                
for d in sorted(os.listdir(env_dir)):
    p = os.path.join(env_dir, d)
    if not os.path.isdir(p):
        continue                                                              
    for v in os.listdir(p):
        if os.path.isdir(os.path.join(p, v)):                                 
            games.append(f"{d}-{v}")                                          
   
print(f"Found {len(games)} games: {games}")                                   
                                            
swarm = Swarm("cognitiveagent", "https://three.arcprize.org", games,          
tags=["kaggle", "cognition-engine"])
scorecard = swarm.main()                                                      
                                            
if scorecard:
    print("\n=== FINAL SCORECARD ===")
    print(json.dumps(scorecard.model_dump(), indent=2))
    print(f"\nOverall Score: {scorecard.score}")   

Found 25 games: ['ar25-0c556536', 'bp35-0a0ad940', 'cd82-fb555c5d', 'cn04-2fe56bfb', 'dc22-fdcac232', 'ft09-0d8bbf25', 'g50t-5849a774', 'ka59-38d34dbb', 'lf52-271a04aa', 'lp85-305b61c3', 'ls20-9607627b', 'm0r0-492f87ba', 'r11l-495a7899', 're86-8af5384d', 's5i5-18d95033', 'sb26-7fbdac44', 'sc25-635fd71a', 'sk48-d8078629', 'sp80-589a99af', 'su15-1944f8ab', 'tn36-ef4dde99', 'tr87-cd924810', 'tu93-0768757b', 'vc33-5430563c', 'wa30-ee6fef47']
INFO:arc_agi.scorecard:Initialized ScorecardManager with idle_for=0:15:00 and max_open_for=3 days, 0:00:00
***** MAKING SCORECARD
2026-04-15 02:47:49 | INFO | Created new scorecard: 99e2a081-af9d-4d9b-9351-8bfb19877d97
2026-04-15 02:47:49,593 | INFO |Created new scorecard: 99e2a081-af9d-4d9b-9351-8bfb19877d97
***** MAKING ALL AGENTS with card id: 99e2a081-af9d-4d9b-9351-8bfb19877d97
2026-04-15 02:47:49 | INFO | Successfully loaded game class Ar25 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/ar25/0c556536/ar25.py
2026-04-15